# UD3.06 — Analisis exploratorio guiado: TechStore 2024

**Modulo 5073 · Programacion de Inteligencia Artificial · Curso 2026/27**
UD3 — NumPy y Pandas · 14 horas

Criterio 2.c · Ensayo guiado antes de la practica P3.2


## CONFIGURACIÓN INICIAL

### Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# La visualización se estudia a fondo en la UD4. Aquí los gráficos son solo
# una herramienta para mirar los datos, y se usan los de pandas.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)

print("numpy", np.__version__, "· pandas", pd.__version__)


### Los datos

Si trabajas en **Colab**, ejecuta la celda siguiente para descargar el fichero de datos.
Si trabajas en **local**, sáltala: el CSV ya está en `datos/`.


In [ ]:
# Solo en Google Colab: descarga el fichero de datos de la unidad
import os, urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD3/datos/")
FICHERO = "ecommerce_ventas_2024.csv"

if not os.path.exists(os.path.join("datos", FICHERO)):
    os.makedirs("datos", exist_ok=True)
    urllib.request.urlretrieve(BASE + FICHERO, os.path.join("datos", FICHERO))
    print("descargado:", FICHERO)
else:
    print("El fichero ya está en datos/, no descargo nada")


---

# FASE 1: PREPARACIÓN Y CARGA DE DATOS (15%)

**Duración estimada:** 1-1.5 horas

### Objetivos de esta fase:
- Cargar y explorar el dataset
- Comprender la estructura y tipos de datos
- Identificar problemas de calidad
- Planificar la estrategia de limpieza

## 1.1 Carga del Dataset

**Tareas:**
1. Cargar el CSV usando Pandas
2. Configurar correctamente los tipos de datos
3. Convertir la columna `Fecha` a datetime

In [ ]:
# TODO: Cargar el dataset
# Sugerencia: df = pd.read_csv('datos/ecommerce_ventas_2024.csv')

df = pd.read_csv('datos/ecommerce_ventas_2024.csv')

# TODO: Convertir columna Fecha a datetime
# Sugerencia: df['Fecha'] = pd.to_datetime(df['Fecha'])

df['Fecha'] = pd.to_datetime(df['Fecha'])

print("✓ Dataset cargado exitosamente")
print(f" Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")

## 1.2 Exploración Inicial

**Tareas:**
1. Mostrar dimensiones del dataset
2. Inspeccionar primeras y últimas filas
3. Generar resumen con `.info()` y `.describe()`
4. Identificar tipos de datos de cada columna

In [ ]:
# Mostrar las primeras 10 filas
print(" PRIMERAS 10 FILAS DEL DATASET:")
print("="*100)
df.head(10)

In [ ]:
# Mostrar las últimas 5 filas
print(" ÚLTIMAS 5 FILAS DEL DATASET:")
print("="*100)
df.tail()

In [ ]:
# Información general del dataset
print("ℹ INFORMACIÓN DEL DATASET:")
print("="*100)
df.info()

In [ ]:
# Estadísticas descriptivas de las columnas numéricas
print(" ESTADÍSTICAS DESCRIPTIVAS (VARIABLES NUMÉRICAS):")
print("="*100)
df.describe()

In [ ]:
# Estadísticas descriptivas de las columnas categóricas
print(" ESTADÍSTICAS DESCRIPTIVAS (VARIABLES CATEGÓRICAS):")
print("="*100)
df.describe(include=['object'])

## 1.3 Identificación de Problemas de Calidad

**Tareas:**
1. Contar valores nulos por columna
2. Identificar registros duplicados
3. Detectar valores atípicos evidentes
4. Documentar problemas encontrados

In [ ]:
# TODO: Análisis de valores nulos
print(" ANÁLISIS DE VALORES NULOS:")
print("="*100)

# Contar nulos por columna
nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

# Crear DataFrame resumen
resumen_nulos = pd.DataFrame({
    'Columna': nulos.index,
    'Valores_Nulos': nulos.values,
    'Porcentaje_%': porcentaje_nulos.values
})

# Filtrar solo columnas con nulos
resumen_nulos = resumen_nulos[resumen_nulos['Valores_Nulos'] > 0].sort_values('Valores_Nulos', ascending=False)

print(resumen_nulos.to_string(index=False))
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")
print(f"Porcentaje total: {(df.isnull().sum().sum() / df.size) * 100:.2f}%")

In [ ]:
# TODO: Análisis de duplicados
print(" ANÁLISIS DE REGISTROS DUPLICADOS:")
print("="*100)

# Contar duplicados (excluyendo ID_Transaccion que es único)
duplicados = df.duplicated(subset=df.columns.difference(['ID_Transaccion']))
n_duplicados = duplicados.sum()

print(f"Registros duplicados: {n_duplicados}")
print(f"Porcentaje: {(n_duplicados/len(df))*100:.2f}%")

if n_duplicados > 0:
    print("\n Ejemplo de registros duplicados:")
    print(df[duplicados].head())

In [ ]:
# TODO: Detectar valores atípicos
print(" ANÁLISIS DE VALORES ATÍPICOS:")
print("="*100)

# Precios negativos
precios_negativos = (df['Precio_Unitario'] < 0).sum()
print(f"\n1. Precios negativos (error de sistema): {precios_negativos}")

if precios_negativos > 0:
    print(" Ejemplos:")
    print(df[df['Precio_Unitario'] < 0][['ID_Transaccion', 'Producto', 'Precio_Unitario']].head())

# Cantidades muy altas
cantidad_alta = (df['Cantidad'] > 10).sum()
print(f"\n2. Cantidades > 10 unidades: {cantidad_alta}")

# Descuentos fuera de rango
descuentos_invalidos = ((df['Descuento_%'] < 0) | (df['Descuento_%'] > 100)).sum()
print(f"\n3. Descuentos fuera de rango [0-100]: {descuentos_invalidos}")

# Outliers de precio usando IQR
Q1 = df['Precio_Unitario'].quantile(0.25)
Q3 = df['Precio_Unitario'].quantile(0.75)
IQR = Q3 - Q1
outliers_precio = ((df['Precio_Unitario'] < (Q1 - 1.5 * IQR)) | (df['Precio_Unitario'] > (Q3 + 1.5 * IQR))).sum()
print(f"\n4. Outliers de precio (método IQR): {outliers_precio}")
print(f" - Q1: {Q1:.2f}€")
print(f" - Q3: {Q3:.2f}€")
print(f" - IQR: {IQR:.2f}€")
print(f" - Límite inferior: {(Q1 - 1.5 * IQR):.2f}€")
print(f" - Límite superior: {(Q3 + 1.5 * IQR):.2f}€")

In [ ]:
# TODO: Análisis de inconsistencias en categorías
print(" ANÁLISIS DE INCONSISTENCIAS:")
print("="*100)

print("\n1. Categorías únicas encontradas:")
print(df['Categoria'].value_counts())

# Detectar inconsistencias (mayúsculas vs minúsculas)
categorias_inconsistentes = df['Categoria'].str.islower().sum()
print(f"\n2. Categorías en minúsculas (inconsistencia): {categorias_inconsistentes}")

## 1.4 Plan de Limpieza de Datos

** DOCUMENTA AQUÍ TU ESTRATEGIA:**

Basándote en el análisis anterior, documenta tu plan de limpieza:

### Valores Nulos:
- **Estrategia para columna X:** [Eliminar / Rellenar con media / Rellenar con moda / etc.]
- **Justificación:** [Explica por qué elegiste esta estrategia]

### Duplicados:
- **Estrategia:** [Eliminar / Mantener primero / Investigar]
- **Justificación:** [Explica tu decisión]

### Outliers:
- **Precios negativos:** [Eliminar / Convertir a positivo / Marcar como error]
- **Outliers de precio:** [Mantener / Eliminar / Transformar]
- **Justificación:** [Explica tu decisión]

### Inconsistencias:
- **Categorías:** [Normalizar a mayúsculas / minúsculas / Title Case]
- **Justificación:** [Explica tu decisión]

---

# FASE 2: ANÁLISIS EXPLORATORIO CON NUMPY (25%)

**Duración estimada:** 2-2.5 horas

### Objetivos de esta fase:
- Realizar análisis estadístico usando NumPy
- Detectar outliers y patrones
- Calcular métricas clave de negocio
- Aplicar operaciones de álgebra lineal cuando sea relevante

## 2.1 Limpieza de Datos

**Implementa tu estrategia de limpieza aquí**

In [ ]:
# TODO: Crear una copia del dataset para preservar el original
df_clean = df.copy()

print(" DATASET ORIGINAL:")
print(f" Filas: {len(df)}")
print(f" Nulos: {df.isnull().sum().sum()}")
print(f" Duplicados: {df.duplicated(subset=df.columns.difference(['ID_Transaccion'])).sum()}")

# TODO: Implementar estrategia de limpieza
# Ejemplo: eliminar duplicados
# df_clean = df_clean.drop_duplicates(subset=df_clean.columns.difference(['ID_Transaccion']))

# Ejemplo: tratar valores nulos
# df_clean['Descuento_%'].fillna(0, inplace=True)

# Ejemplo: corregir precios negativos
# df_clean = df_clean[df_clean['Precio_Unitario'] > 0]

# Ejemplo: normalizar categorías
# df_clean['Categoria'] = df_clean['Categoria'].str.title()

print("\n✓ DATASET LIMPIO:")
print(f" Filas: {len(df_clean)}")
print(f" Filas eliminadas: {len(df) - len(df_clean)}")

In [ ]:
# TODO: Crear columna calculada Total_Venta
# Fórmula: Total_Venta = (Precio_Unitario * Cantidad) * (1 - Descuento_%/100) + Costo_Envio

df_clean['Total_Venta'] = (
    df_clean['Precio_Unitario'] * df_clean['Cantidad'] * (1 - df_clean['Descuento_%']/100)
    + df_clean['Costo_Envio']
)

print("✓ Columna 'Total_Venta' creada")
print(f"\nEstadísticas de Total_Venta:")
print(df_clean['Total_Venta'].describe())

## 2.2 Estadísticas Descriptivas con NumPy

**Calcula estadísticas usando operaciones de NumPy**

In [ ]:
# TODO: Convertir columnas a arrays de NumPy para análisis
precios = df_clean['Precio_Unitario'].values
cantidades = df_clean['Cantidad'].values
total_ventas = df_clean['Total_Venta'].values
descuentos = df_clean['Descuento_%'].values

print(" ESTADÍSTICAS DESCRIPTIVAS CON NUMPY:")
print("="*100)

# Precio Unitario
print("\n1. PRECIO UNITARIO:")
print(f" Media: {np.mean(precios):.2f}€")
print(f" Mediana: {np.median(precios):.2f}€")
print(f" Desviación estándar: {np.std(precios):.2f}€")
print(f" Mínimo: {np.min(precios):.2f}€")
print(f" Máximo: {np.max(precios):.2f}€")
print(f" Percentil 25: {np.percentile(precios, 25):.2f}€")
print(f" Percentil 75: {np.percentile(precios, 75):.2f}€")

# TODO: Hacer lo mismo para Cantidad, Total_Venta y Descuento
# [COMPLETAR CÓDIGO AQUÍ]

## 2.3 Detección de Outliers

**Implementa métodos de detección de outliers**

In [ ]:
# TODO: Método IQR para detectar outliers en Total_Venta
print(" DETECCIÓN DE OUTLIERS (MÉTODO IQR):")
print("="*100)

Q1 = np.percentile(total_ventas, 25)
Q3 = np.percentile(total_ventas, 75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_iqr = (total_ventas < limite_inferior) | (total_ventas > limite_superior)

print(f"Q1: {Q1:.2f}€")
print(f"Q3: {Q3:.2f}€")
print(f"IQR: {IQR:.2f}€")
print(f"Límite inferior: {limite_inferior:.2f}€")
print(f"Límite superior: {limite_superior:.2f}€")
print(f"\nOutliers detectados: {np.sum(outliers_iqr)} ({(np.sum(outliers_iqr)/len(total_ventas))*100:.2f}%)")

In [ ]:
# Metodo del z: cuantas desviaciones tipicas se separa cada valor de la media.
#
# La receta cabe en dos lineas de NumPy y conviene escribirla a mano una vez,
# porque asi se ve que el criterio "|z| > 3" es una decision tuya y no una
# propiedad de los datos: con una distribucion muy asimetrica, como la de los
# precios de un catalogo, marca como anomalo lo que solo es caro.
z = (precios - precios.mean()) / precios.std(ddof=0)
outliers_z = np.abs(z) > 3

print("DETECCION DE VALORES ANOMALOS (METODO DEL Z):")
print("=" * 70)
print(f"Detectados (|z| > 3): {outliers_z.sum()} "
      f"({outliers_z.sum() / len(precios) * 100:.2f} %)")

if outliers_z.any():
    print("\nPrecios marcados como anomalos:")
    print(f"  Minimo: {precios[outliers_z].min():.2f} EUR")
    print(f"  Maximo: {precios[outliers_z].max():.2f} EUR")
    print("\nCompara este recuento con el que da el metodo del rango")
    print("intercuartilico de la celda anterior, y razona en la memoria")
    print("cual de los dos criterios describe mejor estos datos.")


## 2.4 Visualizaciones de Distribuciones

In [ ]:
# TODO: Crear histogramas de variables clave
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Precio Unitario
axes[0, 0].hist(precios, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribución de Precio Unitario', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Precio (€)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].axvline(np.mean(precios), color='red', linestyle='--', label=f'Media: {np.mean(precios):.2f}€')
axes[0, 0].axvline(np.median(precios), color='green', linestyle='--', label=f'Mediana: {np.median(precios):.2f}€')
axes[0, 0].legend()

# TODO: Completar para Cantidad, Total_Venta y Descuento
# [COMPLETAR CÓDIGO AQUÍ]

plt.tight_layout()
plt.show()

## 2.5 Matriz de Correlación

In [ ]:
# TODO: Calcular matriz de correlación
columnas_numericas = ['Precio_Unitario', 'Cantidad', 'Descuento_%', 'Costo_Envio', 'Total_Venta']
correlacion = df_clean[columnas_numericas].corr()

print(" MATRIZ DE CORRELACIÓN:")
print("="*100)
print(correlacion)

# Un mapa de calor de una tabla pivote es, literalmente, dibujar la matriz.
# Aqui se hace con matplotlib a pelo: la visualizacion se estudia en la UD4
# y no merece la pena arrastrar una biblioteca mas solo para esto.
tabla = correlacion
fig, ax = plt.subplots(figsize=(10, 6))
imagen = ax.imshow(tabla.to_numpy(dtype=float), aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(tabla.shape[1]), tabla.columns, rotation=45, ha="right")
ax.set_yticks(range(tabla.shape[0]), tabla.index)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        ax.text(j, i, f"{tabla.iat[i, j]:,.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagen, ax=ax, shrink=0.8)
ax.set_title("Matriz de correlacion de las variables numericas")
fig.tight_layout()
plt.show()


---

# FASE 3: MANIPULACIÓN AVANZADA CON PANDAS (30%)

**Duración estimada:** 2.5-3 horas

### Objetivos de esta fase:
- Aplicar operaciones avanzadas de Pandas
- Realizar análisis por segmentos (GroupBy)
- Generar insights de negocio

## 3.1 Agrupaciones y Agregaciones

**Analiza ventas por diferentes dimensiones**

In [ ]:
# TODO: Ventas totales por producto
print(" TOP 10 PRODUCTOS MÁS VENDIDOS (POR INGRESOS):")
print("="*100)

ventas_por_producto = df_clean.groupby('Producto').agg({
    'Total_Venta': 'sum',
    'Cantidad': 'sum',
    'ID_Transaccion': 'count'
}).rename(columns={'ID_Transaccion': 'N_Transacciones'})

ventas_por_producto = ventas_por_producto.sort_values('Total_Venta', ascending=False)

print(ventas_por_producto.head(10))

# Visualizar
plt.figure(figsize=(12, 6))
ventas_por_producto.head(10)['Total_Venta'].plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Top 10 Productos por Ingresos Totales', fontsize=14, fontweight='bold')
plt.xlabel('Producto')
plt.ylabel('Ingresos Totales (€)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Ventas por región
print(" VENTAS POR REGIÓN:")
print("="*100)

# [COMPLETAR CÓDIGO AQUÍ]
# Sugerencia: df_clean.groupby('Region')

In [ ]:
# TODO: Ventas por categoría
print(" VENTAS POR CATEGORÍA:")
print("="*100)

# [COMPLETAR CÓDIGO AQUÍ]

## 3.2 Análisis de Clientes (RFM Simplificado)

**RFM:** Recency, Frequency, Monetary

In [ ]:
# TODO: Calcular RFM para cada cliente
print(" ANÁLISIS RFM DE CLIENTES:")
print("="*100)

# Fecha de referencia (última fecha del dataset + 1 día)
fecha_referencia = df_clean['Fecha'].max() + pd.Timedelta(days=1)

# Calcular RFM
rfm = df_clean.groupby('Cliente_ID').agg({
    'Fecha': lambda x: (fecha_referencia - x.max()).days, # Recency
    'ID_Transaccion': 'count', # Frequency
    'Total_Venta': 'sum' # Monetary
}).rename(columns={
    'Fecha': 'Recency_dias',
    'ID_Transaccion': 'Frequency',
    'Total_Venta': 'Monetary'
})

print("\nEstadísticas RFM:")
print(rfm.describe())

print("\n TOP 10 CLIENTES VIP (Por Monetary):")
print(rfm.sort_values('Monetary', ascending=False).head(10))

In [ ]:
# TODO: Segmentar clientes
def segmentar_cliente(row):
    """
    Segmenta clientes en:
    - VIP: Alta frecuencia (>3) y alto gasto (>P75)
    - Frecuentes: Alta frecuencia (>3)
    - Perdidos: Alta recencia (>180 días)
    - Ocasionales: Resto
    """
    if row['Frequency'] > 3 and row['Monetary'] > rfm['Monetary'].quantile(0.75):
        return 'VIP'
    elif row['Frequency'] > 3:
        return 'Frecuentes'
    elif row['Recency_dias'] > 180:
        return 'Perdidos'
    else:
        return 'Ocasionales'

rfm['Segmento'] = rfm.apply(segmentar_cliente, axis=1)

print(" DISTRIBUCIÓN DE SEGMENTOS DE CLIENTES:")
print("="*100)
print(rfm['Segmento'].value_counts())

# Visualizar
plt.figure(figsize=(10, 6))
rfm['Segmento'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90, colors=['gold', 'lightblue', 'lightcoral', 'lightgreen'])
plt.title('Distribución de Segmentos de Clientes', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 3.3 Pivot Tables y Crosstabs

In [ ]:
# TODO: Crear tabla dinámica: Categoría × Región
print(" TABLA DINÁMICA: VENTAS POR CATEGORÍA Y REGIÓN:")
print("="*100)

pivot_cat_region = pd.pivot_table(
    df_clean,
    values='Total_Venta',
    index='Categoria',
    columns='Region',
    aggfunc='sum',
    margins=True,
    margins_name='TOTAL'
)

print(pivot_cat_region)

# Un mapa de calor de una tabla pivote es, literalmente, dibujar la matriz.
# Aqui se hace con matplotlib a pelo: la visualizacion se estudia en la UD4
# y no merece la pena arrastrar una biblioteca mas solo para esto.
tabla = pivot_cat_region.iloc[:-1, :-1]  # sin la fila y columna de totales
fig, ax = plt.subplots(figsize=(10, 6))
imagen = ax.imshow(tabla.to_numpy(dtype=float), aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(tabla.shape[1]), tabla.columns, rotation=45, ha="right")
ax.set_yticks(range(tabla.shape[0]), tabla.index)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        ax.text(j, i, f"{tabla.iat[i, j]:,.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagen, ax=ax, shrink=0.8)
ax.set_title("Ventas por categoria y region (EUR)")
fig.tight_layout()
plt.show()


## 3.4 Series Temporales

**Analiza tendencias y estacionalidad**

In [ ]:
# TODO: Crear features temporales
df_clean['Mes'] = df_clean['Fecha'].dt.month
df_clean['Mes_Nombre'] = df_clean['Fecha'].dt.strftime('%B')
df_clean['Dia_Semana'] = df_clean['Fecha'].dt.day_name()
df_clean['Trimestre'] = df_clean['Fecha'].dt.quarter

print("✓ Features temporales creadas: Mes, Dia_Semana, Trimestre")

In [ ]:
# TODO: Ventas por mes
print(" VENTAS MENSUALES 2024:")
print("="*100)

ventas_mensuales = df_clean.groupby('Mes')['Total_Venta'].agg(['sum', 'count', 'mean'])
ventas_mensuales.columns = ['Ingresos_Totales', 'N_Transacciones', 'Ticket_Promedio']

print(ventas_mensuales)

# Visualizar
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(ventas_mensuales.index, ventas_mensuales['Ingresos_Totales'],
         marker='o', linewidth=2, color='steelblue', label='Ingresos Totales')
ax1.set_xlabel('Mes', fontsize=12)
ax1.set_ylabel('Ingresos Totales (€)', fontsize=12, color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_xticks(range(1, 13))
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.bar(ventas_mensuales.index, ventas_mensuales['N_Transacciones'],
        alpha=0.3, color='coral', label='N° Transacciones')
ax2.set_ylabel('N° Transacciones', fontsize=12, color='coral')
ax2.tick_params(axis='y', labelcolor='coral')

plt.title('Evolución de Ventas Mensuales 2024', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Ventas por día de la semana
print(" VENTAS POR DÍA DE LA SEMANA:")
print("="*100)

# [COMPLETAR CÓDIGO AQUÍ]
# Sugerencia: df_clean.groupby('Dia_Semana')

---

# FASE 4: INSIGHTS Y CONCLUSIONES (20%)

**Duración estimada:** 1.5-2 horas

### Objetivos de esta fase:
- Sintetizar hallazgos
- Generar insights de negocio accionables
- Crear visualizaciones de alto impacto
- Responder preguntas de negocio

## 4.1 Responder Preguntas de Negocio Clave

### Pregunta 1: ¿Qué productos deberíamos priorizar en nuestro inventario?

**TODO: Analiza y responde aquí**

In [ ]:
# TODO: Identificar top 5 productos por volumen e ingresos
# Analizar estabilidad de ventas (varianza)
# Recomendar niveles de stock

# [COMPLETAR CÓDIGO AQUÍ]

** RESPUESTA:**

[Escribe tu respuesta aquí basándote en el análisis]

### Pregunta 2: ¿En qué regiones deberíamos invertir más en marketing?

**TODO: Analiza y responde aquí**

In [ ]:
# TODO: Comparar ventas actuales por región
# Identificar regiones con mayor potencial
# Calcular ticket promedio por región

# [COMPLETAR CÓDIGO AQUÍ]

** RESPUESTA:**

[Escribe tu respuesta aquí]

### Pregunta 3-5: [CONTINÚA CON LAS DEMÁS PREGUNTAS]

- Pregunta 3: ¿Qué productos tienen margen para aumentar precio?
- Pregunta 4: ¿Cómo podemos mejorar la retención de clientes?
- Pregunta 5: ¿Cuándo deberíamos prepararnos para picos de demanda?

## 4.2 Insights Clave (Mínimo 8)

**Documenta tus principales hallazgos:**

### Insight 1:
**Hallazgo:** [Describe el hallazgo con datos específicos]

**Implicación:** [Qué significa esto para el negocio]

**Acción recomendada:** [Qué hacer al respecto]

---

### Insight 2:
...

[CONTINÚA HASTA AL MENOS 8 INSIGHTS]

## 4.3 Dashboard Visual

**Crea 8-12 visualizaciones clave**

In [ ]:
# TODO: Crear dashboard con las visualizaciones más importantes

# Ejemplo de estructura:
fig, axes = plt.subplots(3, 3, figsize=(20, 15))

# Gráfico 1: Top productos
# Gráfico 2: Ventas por región
# Gráfico 3: Evolución temporal
# Gráfico 4: Distribución de descuentos
# Gráfico 5: Segmentos de clientes
# etc.

# [COMPLETAR CÓDIGO AQUÍ]

plt.tight_layout()
plt.show()

## 4.4 Recomendaciones Priorizadas

### Recomendación 1 (ALTA PRIORIDAD):
**Acción:** [Qué hacer]

**Justificación:** [Por qué]

**Impacto esperado:** [Qué resultados esperar]

**Métrica de éxito:** [Cómo medir el éxito]

---

### Recomendación 2-5:
[CONTINÚA CON AL MENOS 5 RECOMENDACIONES]

---

# FASE 5: DOCUMENTACIÓN Y CONCLUSIONES FINALES (10%)

**Duración estimada:** 1 hora

## Resumen Ejecutivo

### Contexto:
[Resume brevemente el proyecto y objetivos]

### Metodología:
[Describe las técnicas utilizadas]

### Principales Hallazgos:
1. [Hallazgo 1]
2. [Hallazgo 2]
3. [Hallazgo 3]

### Recomendaciones Clave:
1. [Recomendación 1]
2. [Recomendación 2]
3. [Recomendación 3]

### Próximos Pasos:
[Qué análisis adicionales se podrían hacer]

## Reflexiones Técnicas

### Desafíos Encontrados:
[Describe los principales desafíos técnicos]

### Decisiones Tomadas:
[Explica las decisiones clave y su justificación]

### Aprendizajes:
[Qué aprendiste durante el proyecto]